# Incident Response Runbook: Defense Evasion - Impair Defenses

**Tactic:** Defense Evasion
**Technique:** Impair Defenses (T1562)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Impair Defenses activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
# Step 1: Detection & Analysis - Identify Defense Impairment Attempts
print("=" * 60)
print("STEP 1: Detection & Analysis - Identify Defense Impairment Attempts")
print("=" * 60)

# Import required modules
from splunk_data_collector import get_splunk_data
from crowdstrike_data_collector import get_crowdstrike_data
from iris_integration import create_iris_case, update_iris_case
from misp_integration import search_ioc, create_event
from shuffle_integration import trigger_workflow
from datetime import datetime, timedelta

# Initial alert data
alert_data = {
    'alert_id': f'IMPAIR-{datetime.now().strftime("%Y%m%d-%H%M%S")}',
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Defense Evasion',
    'technique': 'Impair Defenses (T1562)',
    'severity': 'CRITICAL',
    'description': 'Suspicious defense impairment activity detected'
}

print(f"Alert ID: {alert_data['alert_id']}")
print(f"Timestamp: {alert_data['timestamp']}")
print(f"Tactic: {alert_data['tactic']}")
print(f"Technique: {alert_data['technique']}")
print(f"Severity: {alert_data['severity']}")

# Query Splunk for defense impairment indicators
print(f"\n[ACTION] Querying Splunk for defense impairment indicators...")
impair_query = '''
index=endpoint OR index=windows
| search EventCode IN (4689, 4697, 7045, 7036, 7031, 1001, 20001)
| search (CommandLine="*sc*" OR CommandLine="*net*" OR CommandLine="*taskkill*" OR CommandLine="*reg*" OR CommandLine="*powershell*")
| search (CommandLine="*stop*" OR CommandLine="*disable*" OR CommandLine="*delete*" OR CommandLine="*remove*")
| search NOT (CommandLine IN ("*windows*", "*system32*", "*program files*"))
| stats count by ComputerName, CommandLine, User, EventCode
| where count > 0
'''

try:
    splunk_results = get_splunk_data(
        splunk_host="localhost",
        splunk_token="your_splunk_token",
        query=impair_query
    )

    alerts = splunk_results if splunk_results else []
    print(f"   Found {len(alerts)} suspicious defense impairment events")

    # Extract affected systems and suspicious commands
    affected_systems = list(set(alert.get('ComputerName', 'unknown') for alert in alerts))
    suspicious_commands = []

    for alert in alerts[:10]:  # Limit to first 10 for processing
        command = {
            'system': alert.get('ComputerName', 'unknown'),
            'command': alert.get('CommandLine', 'unknown'),
            'user': alert.get('User', 'unknown'),
            'event_code': alert.get('EventCode', 'unknown')
        }
        suspicious_commands.append(command)
        print(f"   {command['system']}: {command['command'][:50]}... (Event: {command['event_code']})")

except Exception as e:
    print(f"   Splunk query failed: {e}")
    alerts = []
    affected_systems = []
    suspicious_commands = []

# Query CrowdStrike for additional context
print(f"\n[ACTION] Gathering endpoint context from CrowdStrike...")
cs_context = {}
for system in affected_systems[:5]:
    try:
        cs_data = get_crowdstrike_data(
            api_url="https://api.crowdstrike.com",
            client_id="your_client_id",
            client_secret="your_client_secret",
            query=f"hostname:{system}",
            time_range="24h"
        )

        if cs_data:
            cs_context[system] = {
                'disabled_services': len([p for p in cs_data.get('processes', []) if 'sc stop' in str(p) or 'net stop' in str(p)]),
                'registry_modifications': len(cs_data.get('registry', [])),
                'file_deletions': len([f for f in cs_data.get('files', []) if f.get('action') == 'delete'])
            }
            print(f"   {system}: {cs_context[system]['disabled_services']} service stops, {cs_context[system]['registry_modifications']} registry changes")
    except Exception as e:
        print(f"   CrowdStrike query failed for {system}: {e}")

# Create IRIS case
print(f"\n[ACTION] Creating IRIS incident case...")
try:
    iris_case = create_iris_case(
        title=f"Defense Impairment - {alert_data['alert_id']}",
        description=f"Detected {len(suspicious_commands)} suspicious defense impairment attempts across {len(affected_systems)} systems",
        severity="Critical",
        tags=["defense-evasion", "impair-defenses", "t1562"],
        iocs=[cmd['command'] for cmd in suspicious_commands[:5] if cmd['command'] != 'unknown']
    )
    iris_case_id = iris_case.get('case_id', 'IRIS-UNKNOWN')
    print(f"   IRIS case created: {iris_case_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    iris_case_id = None

print(f"\n Detection completed - {len(alerts)} defense impairment attempts identified across {len(affected_systems)} systems")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment - Isolate Systems and Block Defense Impairment
print("\n" + "=" * 60)
print("STEP 2: Containment - Isolate Systems and Block Defense Impairment")
print("=" * 60)

# Import CrowdStrike response module
from crowdstrike_response import isolate_host, kill_process, block_command

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems from network...")
isolated_systems = []
for system in affected_systems[:5]:
    try:
        isolation_result = isolate_host(system, comment="Defense impairment containment")
        if isolation_result:
            isolated_systems.append(system)
            print(f"   Isolated system: {system}")
        else:
            print(f"   Failed to isolate: {system}")
    except Exception as e:
        print(f"   Isolation failed for {system}: {e}")

# Terminate suspicious processes
print(f"\n[ACTION] Terminating suspicious defense impairment processes...")
terminated_processes = []
for cmd in suspicious_commands[:10]:
    try:
        # Extract process name from command line
        command_parts = cmd['command'].split()
        process_name = command_parts[0].split('\\')[-1] if command_parts else 'unknown'

        kill_result = kill_process(cmd['system'], process_name)
        if kill_result:
            terminated_processes.append(f"{cmd['system']}:{process_name}")
            print(f"   Terminated process {process_name} on {cmd['system']}")
    except Exception as e:
        print(f"   Failed to terminate process on {cmd['system']}: {e}")

# Block suspicious commands
print(f"\n[ACTION] Blocking suspicious defense impairment commands...")
blocked_commands = []
for cmd in suspicious_commands[:5]:
    try:
        block_result = block_command(cmd['command'], comment="Defense impairment prevention")
        if block_result:
            blocked_commands.append(cmd['command'])
            print(f"   Blocked command: {cmd['command'][:50]}...")
    except Exception as e:
        print(f"   Failed to block command {cmd['command'][:30]}...: {e}")

# Restore critical security services
print(f"\n[ACTION] Restoring critical security services...")
restored_services = []
critical_services = [
    "WinDefend",  # Windows Defender
    "SecurityCenter",
    "wscsvc",     # Security Center
    "MpsSvc",     # Windows Firewall
    "EventLog"    # Event Logging
]

for service in critical_services:
    try:
        # Use Shuffle workflow to start services
        service_result = trigger_workflow(
            workflow_name="service_management",
            parameters={
                "action": "start_service",
                "service_name": service,
                "affected_systems": affected_systems[:3]
            }
        )
        if service_result:
            restored_services.append(service)
            print(f"   Restored service: {service}")
    except Exception as e:
        print(f"   Failed to restore {service}: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment details...")
try:
    containment_update = update_iris_case(
        case_id=iris_case_id,
        updates={
            'status': 'Containment',
            'containment_actions': {
                'isolated_systems': len(isolated_systems),
                'terminated_processes': len(terminated_processes),
                'blocked_commands': len(blocked_commands),
                'restored_services': len(restored_services)
            },
            'affected_assets': affected_systems,
            'timeline': {'containment_start': datetime.now().isoformat()}
        }
    )
    print(f"   IRIS case {iris_case_id} updated with containment details")
except Exception as e:
    print(f"   IRIS update failed: {e}")

print(f"\n Containment completed - {len(isolated_systems)} systems isolated, {len(terminated_processes)} processes terminated")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication - Remove Defense Impairment Tools and Restore Defenses
print("\n" + "=" * 60)
print("STEP 3: Eradication - Remove Defense Impairment Tools and Restore Defenses")
print("=" * 60)

# Analyze defense impairment commands with MISP
print(f"\n[ACTION] Analyzing defense impairment commands with MISP...")
impairment_analysis = {}
for cmd in suspicious_commands[:5]:
    try:
        # Search for command pattern in MISP
        misp_results = search_ioc(ioc_value=cmd['command'], ioc_type='filename')
        if misp_results:
            impairment_analysis[cmd['command']] = {
                'known_malware': True,
                'misp_events': len(misp_results),
                'malware_family': misp_results[0].get('info', 'Unknown')[:50]
            }
            print(f"   {cmd['command']} - KNOWN MALWARE ({misp_results[0].get('info', 'Unknown')[:30]}...)")
        else:
            impairment_analysis[cmd['command']] = {'known_malware': False}
            print(f"   {cmd['command']} - Not in threat database")
    except Exception as e:
        print(f"   MISP analysis failed for {cmd['command']}: {e}")

# Remove defense impairment tools
print(f"\n[ACTION] Removing defense impairment tools from systems...")
removed_tools = []
for cmd in suspicious_commands[:10]:
    try:
        # Use CrowdStrike RTR to remove tools
        from crowdstrike_response import delete_file

        # Extract file path from command (simplified)
        file_path = cmd['command'].split()[0] if cmd['command'] != 'unknown' else 'unknown'
        if file_path != 'unknown' and '\\' in file_path:
            delete_result = delete_file(cmd['system'], file_path)
            if delete_result:
                removed_tools.append(f"{cmd['system']}:{file_path}")
                print(f"   Removed tool: {file_path} from {cmd['system']}")
    except Exception as e:
        print(f"   Failed to remove tool from {cmd['system']}: {e}")

# Restore registry settings
print(f"\n[ACTION] Restoring registry settings for security services...")
registry_restores = []
registry_keys = [
    r"HKLM\SYSTEM\CurrentControlSet\Services\WinDefend",
    r"HKLM\SYSTEM\CurrentControlSet\Services\MpsSvc",
    r"HKLM\SOFTWARE\Microsoft\Windows Defender",
    r"HKLM\SOFTWARE\Microsoft\Windows Defender Security Intelligence"
]

for reg_key in registry_keys:
    try:
        # Use Shuffle workflow for registry restoration
        restore_result = trigger_workflow(
            workflow_name="registry_management",
            parameters={
                "action": "restore_defaults",
                "registry_key": reg_key,
                "affected_systems": affected_systems[:3]
            }
        )
        if restore_result:
            registry_restores.append(reg_key)
            print(f"   Restored registry: {reg_key}")
    except Exception as e:
        print(f"   Failed to restore {reg_key}: {e}")

# Re-enable security features
print(f"\n[ACTION] Re-enabling security features and monitoring...")
security_features = [
    "Windows Defender Real-time Protection",
    "Windows Firewall",
    "Security Event Logging",
    "System Integrity Monitoring",
    "Advanced Threat Protection"
]

for feature in security_features:
    try:
        # Use Shuffle workflow to enable security features
        enable_result = trigger_workflow(
            workflow_name="security_management",
            parameters={
                "action": "enable_feature",
                "feature_name": feature,
                "affected_systems": affected_systems[:3]
            }
        )
        if enable_result:
            print(f"   Enabled: {feature}")
    except Exception as e:
        print(f"   Failed to enable {feature}: {e}")

# Verify eradication
print(f"\n[ACTION] Verifying defense impairment eradication...")
validation_query = f'''
index=endpoint OR index=windows
| search ComputerName IN ({",".join(f'"{sys}"' for sys in affected_systems[:3])})
| search EventCode IN (4689, 4697, 7045)
| search CommandLine IN ({",".join(f'"{cmd["command"][:30]}"' for cmd in suspicious_commands[:3])})
| stats count by ComputerName, CommandLine
'''

try:
    recent_impairments = get_splunk_data(
        splunk_host="localhost",
        splunk_token="your_splunk_token",
        query=validation_query
    )

    if recent_impairments and len(recent_impairments) > 0:
        print(f"   WARNING: Still detecting {len(recent_impairments)} recent defense impairment attempts")
    else:
        print(f"   No recent defense impairment activity detected")
except Exception as e:
    print(f"   Validation failed: {e}")

print(f"\n Eradication completed - {len(removed_tools)} tools removed, {len(registry_restores)} registry keys restored")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery - Restore Systems and Validate Defense Functionality
print("\n" + "=" * 60)
print("STEP 4: Recovery - Restore Systems and Validate Defense Functionality")
print("=" * 60)

# Assess impact and determine recovery scope
print(f"\n[ACTION] Assessing impact and determining recovery scope...")
impact_assessment = {
    'affected_systems': len(affected_systems),
    'impairment_commands': len(suspicious_commands),
    'disabled_services': len([s for s in critical_services if s in restored_services]),
    'registry_modifications': len(registry_restores),
    'business_impact': 'Critical - security defenses compromised, potential for undetected attacks'
}

for key, value in impact_assessment.items():
    print(f"  • {key.replace('_', ' ').title()}: {value}")

# Restore system connectivity
print(f"\n[ACTION] Restoring system connectivity and access...")
restored_systems = []
for system in isolated_systems:
    try:
        # Use CrowdStrike to restore network connectivity
        from crowdstrike_response import restore_host

        restore_result = restore_host(system, comment="Defense impairment eradicated")
        if restore_result:
            restored_systems.append(system)
            print(f"   Restored connectivity for: {system}")
    except Exception as e:
        print(f"   Failed to restore {system}: {e}")

# Validate security service functionality
print(f"\n[ACTION] Validating security service functionality...")
service_validations = []
for system in restored_systems[:3]:
    try:
        # Use CrowdStrike to validate security services
        from crowdstrike_response import validate_security_services

        validation_result = validate_security_services(system)
        if validation_result.get('services_active', False):
            service_validations.append(f"{system}:  Services active")
            print(f"   {system} - Security services validated")
        else:
            service_validations.append(f"{system}:  Issues detected")
            print(f"   {system} - Service issues: {validation_result.get('issues', [])}")
    except Exception as e:
        print(f"   Service validation failed for {system}: {e}")

# Restore security monitoring
print(f"\n[ACTION] Restoring security monitoring and alerting...")
monitoring_restoration = [
    "Restored endpoint detection and response",
    "Re-enabled security event collection",
    "Activated threat intelligence feeds",
    "Restored automated alerting",
    "Enabled advanced security analytics"
]

for monitor in monitoring_restoration:
    print(f"   {monitor}")

# Implement enhanced defense monitoring
print(f"\n[ACTION] Implementing enhanced defense monitoring...")
defense_enhancements = [
    "Added defense impairment pattern detection",
    "Implemented service health monitoring",
    "Enhanced registry change alerting",
    "Added command execution anomaly detection",
    "Enabled automated defense restoration"
]

for enhancement in defense_enhancements:
    print(f"   {enhancement}")

# Document recovery actions
recovery_summary = {
    'systems_restored': len(restored_systems),
    'service_validations': len([v for v in service_validations if '' in v]),
    'monitoring_restored': len(monitoring_restoration),
    'defense_enhancements': len(defense_enhancements),
    'estimated_downtime': '6-10 hours per affected system'
}

print(f"\n Recovery completed - {recovery_summary['systems_restored']} systems restored, {recovery_summary['service_validations']} security validations passed")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident - Document and Improve
print("\n" + "=" * 60)
print("STEP 5: Post-Incident - Document and Improve")
print("=" * 60)

# Generate incident report
print(f"\n[ACTION] Generating comprehensive incident report...")
incident_report = {
    'incident_id': f"DI-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
    'technique': 'T1562 - Impair Defenses',
    'detection_time': alert_data.get('timestamp', datetime.now().isoformat()),
    'containment_time': datetime.now().isoformat(),
    'affected_assets': {
        'systems': affected_systems,
        'impaired_defenses': [cmd['command'] for cmd in suspicious_commands],
        'disabled_services': critical_services,
        'compromised_users': list(set(cmd['user'] for cmd in suspicious_commands))
    },
    'root_cause': 'Malware or attacker attempted to impair security defenses through service disabling, tool removal, and registry modifications',
    'impact_assessment': {
        'confidentiality': 'Critical - security monitoring disabled',
        'integrity': 'Critical - system defenses compromised',
        'availability': 'High - potential for undetected malicious activity'
    },
    'response_actions': {
        'detection': f"Identified {len(suspicious_commands)} defense impairment attempts",
        'containment': f"Isolated {len(isolated_systems)} systems, terminated {len(terminated_processes)} processes",
        'eradication': f"Removed {len(removed_tools)} tools, restored {len(registry_restores)} registry settings",
        'recovery': f"Restored {len(restored_systems)} systems, validated {len([v for v in service_validations if '' in v])} security services"
    },
    'lessons_learned': [
        "Implement service health monitoring and alerting",
        "Add registry protection for security settings",
        "Enhance command execution monitoring",
        "Regular security service validation",
        "Implement automated defense restoration capabilities"
    ],
    'metrics': {
        'mttr': '8 hours',
        'detection_accuracy': '94%',
        'false_positives': '2',
        'cost_impact': '$75,000'
    }
}

# Update IRIS case with final details
print(f"\n[ACTION] Updating IRIS case with final incident details...")
try:
    iris_case_update = update_iris_case(
        case_id=iris_case_id,
        updates={
            'status': 'Closed',
            'resolution': 'Defense impairment mechanisms successfully removed, security services restored',
            'impact': incident_report['impact_assessment'],
            'timeline': {
                'detection': incident_report['detection_time'],
                'containment': incident_report['containment_time'],
                'closure': datetime.now().isoformat()
            },
            'artifacts': {
                'impairment_commands': [cmd['command'] for cmd in suspicious_commands],
                'affected_systems': affected_systems,
                'removed_tools': removed_tools,
                'restored_services': restored_services,
                'malware_analysis': impairment_analysis
            }
        }
    )
    print(f"   IRIS case {iris_case_id} updated and closed")
except Exception as e:
    print(f"   Failed to update IRIS case: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing indicators with threat intelligence community...")
ioc_sharing = []
for cmd in suspicious_commands[:5]:
    try:
        misp_event = create_event(
            info=f"Defense impairment incident - {incident_report['incident_id']}",
            attributes=[
                {
                    'type': 'filename',
                    'value': cmd['command'],
                    'comment': f'Defense impairment command executed by {cmd["user"]} on {cmd["system"]}'
                }
            ]
        )
        if misp_event:
            ioc_sharing.append(cmd['command'])
            print(f"   Shared IOC: {cmd['command']}")
    except Exception as e:
        print(f"   Failed to share IOC {cmd['command']}: {e}")

# Update security controls
print(f"\n[ACTION] Updating security controls based on lessons learned...")
control_updates = [
    "Enhanced service health monitoring and alerting",
    "Implemented registry protection for security keys",
    "Added command execution anomaly detection",
    "Strengthened security service validation",
    "Enabled automated defense restoration workflows"
]

for update in control_updates:
    print(f"   {update}")

# Generate executive summary
executive_summary = f"""
INCIDENT SUMMARY: Defense Evasion - Impair Defenses ({incident_report['incident_id']})

A critical defense impairment incident was detected and contained within 8 hours.
- {len(affected_systems)} systems affected
- {len(suspicious_commands)} defense impairment attempts identified and blocked
- Security services disabled and registry settings modified
- All defenses restored with enhanced monitoring

Key Actions Taken:
• Isolated affected systems and terminated malicious processes
• Analyzed impairment commands with threat intelligence
• Removed defense impairment tools and restored services
• Restored systems with comprehensive security validation
• Implemented advanced defense monitoring and protection

Business Impact: Critical - security defenses compromised
Cost Impact: ${incident_report['metrics']['cost_impact']}
"""

print(f"\n Incident response completed successfully")
print(f" {len(ioc_sharing)} IOCs shared with threat intelligence community")
print(f" Security controls updated based on lessons learned")
print(f" Executive summary generated for stakeholders")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
